### Mencoba LKP 11

import dulu datanya

In [ ]:
import heapq
import pandas as pd
from IPython.display import display

# Data dari gambar soal (W_max = 11)
W_max = 11
raw_data = [
    {"Item": "A", "W": 5, "V": 34},
    {"Item": "B", "W": 2, "V": 11},
    {"Item": "C", "W": 3, "V": 20},
    {"Item": "D", "W": 7, "V": 50},
    {"Item": "E", "W": 3, "V": 20},
]

# Buat DataFrame dan hitung rasio V/W
df_items = pd.DataFrame(raw_data)
df_items["V/W"] = df_items["V"] / df_items["W"]

# Urutkan berdasarkan rasio V/W tertinggi ke terendah
df_items = df_items.sort_values(by="V/W", ascending=False).reset_index(drop=True)

print("1. Tabel Item Terurut Berdasarkan Rasio V/W:")
display(df_items.style.format({"V/W": "{:.2f}"}))

: 

Fungsi Perhitungan Bound dan Algoritma Branch & Bound

In [ ]:
class Node:
    def __init__(self, level, profit, weight, bound, items, node_id, parent_id, action):
        self.level = level
        self.profit = profit
        self.weight = weight
        self.bound = bound
        self.items = items
        self.node_id = node_id
        self.parent_id = parent_id
        self.action = action

    # Menggunakan Max-Heap berdasarkan nilai bound (negatifkan nilai bound untuk heapq Python)
    def __lt__(self, other):
        return self.bound > other.bound


def calculate_bound(level, profit, weight, W, items_df):
    if weight >= W:
        return 0.0

    profit_bound = float(profit)
    j = level + 1
    tot_weight = weight
    n = len(items_df)

    # Ambil item berikutnya secara utuh selama masih muat
    while j < n and tot_weight + items_df.loc[j, "W"] <= W:
        tot_weight += items_df.loc[j, "W"]
        profit_bound += items_df.loc[j, "V"]
        j += 1

    # Jika masih ada sisa kapasitas, ambil pecahan dari item selanjutnya
    if j < n:
        profit_bound += (W - tot_weight) * items_df.loc[j, "V/W"]

    return profit_bound


def solve_knapsack_bnb(W, items_df):
    n = len(items_df)
    pq = []
    logs = []
    tree_nodes = []

    node_counter = 0
    root_bound = calculate_bound(-1, 0, 0, W, items_df)
    root = Node(-1, 0, 0, root_bound, [], node_counter, None, "Root")
    heapq.heappush(pq, root)
    tree_nodes.append((root, "Root"))

    max_profit = 0
    best_items = []
    step = 1

    while pq:
        u = heapq.heappop(pq)

        # Cek apakah node perlu dipangkas sebelum diekspansi
        if u.bound <= max_profit:
            logs.append({
                "Langkah": step,
                "Node ID": f"N{u.node_id}",
                "Item Terpilih": ", ".join(u.items) if u.items else "-",
                "Berat (W)": u.weight,
                "Nilai (V)": u.profit,
                "Upper Bound": round(u.bound, 2),
                "MaxProfit Sementara": max_profit,
                "Status": f"Dipangkas (Bound <= {max_profit})"
            })
            step += 1
            continue

        logs.append({
            "Langkah": step,
            "Node ID": f"N{u.node_id}",
            "Item Terpilih": ", ".join(u.items) if u.items else "-",
            "Berat (W)": u.weight,
            "Nilai (V)": u.profit,
            "Upper Bound": round(u.bound, 2),
            "MaxProfit Sementara": max_profit,
            "Status": "Diekspansi"
        })
        step += 1

        if u.level == n - 1:
            continue

        next_lvl = u.level + 1
        item_name = items_df.loc[next_lvl, "Item"]
        item_w = int(items_df.loc[next_lvl, "W"])
        item_v = int(items_df.loc[next_lvl, "V"])

        # --- 1. CABANG KIRI: AMBIL ITEM ---
        node_counter += 1
        left_w = u.weight + item_w
        left_v = u.profit + item_v
        left_items = u.items + [item_name]

        if left_w <= W:
            if left_v > max_profit:
                max_profit = left_v
                best_items = left_items

            left_bound = calculate_bound(next_lvl, left_v, left_w, W, items_df)
            left_node = Node(next_lvl, left_v, left_w, left_bound, left_items, node_counter, u.node_id, f"Ambil {item_name}")

            if left_bound > max_profit:
                heapq.heappush(pq, left_node)
                tree_nodes.append((left_node, "Aktif"))
            else:
                tree_nodes.append((left_node, f"Dipangkas (Bound <= {max_profit})"))
        else:
            over_node = Node(next_lvl, left_v, left_w, 0, left_items, node_counter, u.node_id, f"Ambil {item_name}")
            tree_nodes.append((over_node, "Dipangkas (Overweight)"))

        # --- 2. CABANG KANAN: LEWATI ITEM ---
        node_counter += 1
        right_w = u.weight
        right_v = u.profit
        right_items = u.items.copy()
        right_bound = calculate_bound(next_lvl, right_v, right_w, W, items_df)
        right_node = Node(next_lvl, right_v, right_w, right_bound, right_items, node_counter, u.node_id, f"Lewati {item_name}")

        if right_bound > max_profit:
            heapq.heappush(pq, right_node)
            tree_nodes.append((right_node, "Aktif"))
        else:
            tree_nodes.append((right_node, f"Dipangkas (Bound <= {max_profit})"))

    return max_profit, best_items, pd.DataFrame(logs), tree_nodes


max_v, chosen_items, df_steps, all_nodes = solve_knapsack_bnb(W_max, df_items)

print("2. Log Penelusuran Node (Best-First Search):")
display(df_steps)

print(f"\n=== HASIL OPTIMAL ===")
print(f"Nilai Maksimal (V) : {max_v}")
print(f"Item Dipilih       : {', '.join(chosen_items)} (Total Berat = {sum(df_items[df_items['Item'].isin(chosen_items)]['W'])})")